# Prototype: jaxopt Beta fitting (vmap over MCMC samples)

Tests whether we can replace `scipy.optimize.minimize` (BFGS) with `jaxopt.LBFGS`
and then `vmap` over S MCMC samples — enabling Option 3 (proper Monte Carlo
marginalisation over y') at low cost.

## Goals
1. Confirm `jaxopt.LBFGS` gives same Beta params as the scipy version on toy data
2. Test that `jax.vmap` over S samples works (no tracing errors)
3. Benchmark: scipy loop vs jaxopt vmap for S=200 samples, J=15 features
4. Compute Option 3 log likelihood (logsumexp over S fits) and compare to Option 1/2

In [1]:
import sys, time
sys.path.insert(0, '.')

import numpy as np
import jax
import jax.numpy as jnp
import jaxopt
from scipy.optimize import minimize as scipy_minimize
from scipy.special import logsumexp

from model_jax import (
    _neg_ll_one_feature,
    _neg_ll_and_grad,
    _LOG_INITS,
    fit_beta_mixtures_all_features,
    beta_mixture_log_likelihood,
)

print(f"JAX version: {jax.__version__}")
print(f"jaxopt version: {jaxopt.__version__ if hasattr(jaxopt, '__version__') else 'installed'}")
print(f"Default backend: {jax.default_backend()}")

JAX version: 0.9.2
jaxopt version: installed
Default backend: cpu


## 1. Toy data
Simulate participant ratings from a known Beta mixture, plus fake pz1 samples.

In [2]:
rng = np.random.default_rng(0)

N = 100   # participants
J = 5     # features (small for prototyping)
S = 200   # MCMC samples to vmap over

# Fake participant ratings: mix of two Beta distributions per feature
# True params: kind-linked ~ Beta(6,2), not-kind-linked ~ Beta(2,6), mixing ~ 0.6
true_pz1 = np.array([0.6, 0.4, 0.7, 0.3, 0.5])
ratings = np.zeros((N, J))
for j in range(J):
    z = rng.binomial(1, true_pz1[j], size=N)
    ratings[:, j] = np.where(z, rng.beta(6, 2, N), rng.beta(2, 6, N))
ratings = np.clip(ratings, 1e-6, 1 - 1e-6)

# Fake MCMC posterior samples of pz1 (S samples per feature)
# In the real model these come from sigmoid(y'_j MCMC samples)
pz1_samples = rng.beta(5, 3, size=(S, J))  # (S, J), roughly centered near true_pz1

print(f"ratings: {ratings.shape}")
print(f"pz1_samples: {pz1_samples.shape}")
print(f"true pz1: {true_pz1}")
print(f"pz1 sample mean: {pz1_samples.mean(axis=0).round(3)}")

ratings: (100, 5)
pz1_samples: (200, 5)
true pz1: [0.6 0.4 0.7 0.3 0.5]
pz1 sample mean: [0.606 0.622 0.623 0.609 0.619]


## 2. Baseline: scipy BFGS (current approach)
Single fit using the posterior mean pz1 (Option 1).

In [3]:
pz1_mean = jnp.array(pz1_samples.mean(axis=0))  # (J,) — Option 1 point estimate
pz1_broadcast = jnp.tile(pz1_mean, (N, 1))       # (N, J)

t0 = time.time()
beta_params_scipy = fit_beta_mixtures_all_features(jnp.array(ratings), pz1_broadcast)
t_scipy = time.time() - t0

print(f"Scipy fit time: {t_scipy:.3f}s")
print(f"Beta params (J, 4) — [α_kl, β_kl, α_nkl, β_nkl]:")
print(np.array(beta_params_scipy).round(3))

ll_scipy = beta_mixture_log_likelihood(jnp.array(ratings), pz1_mean, beta_params_scipy)
print(f"\nLog likelihood (Option 1, scipy): {ll_scipy:.2f}")

Scipy fit time: 0.691s
Beta params (J, 4) — [α_kl, β_kl, α_nkl, β_nkl]:
[[ 3.695  1.788  3.174 15.452]
 [ 3.504  1.576  3.05  11.231]
 [11.006  2.677  1.857  3.981]
 [ 8.992  2.769  2.466  7.817]
 [ 2.189  1.369  3.183 12.779]]

Log likelihood (Option 1, scipy): 52.91


## 3. jaxopt LBFGS: single fit
Verify jaxopt gives same result as scipy for one feature.

In [4]:
def fit_one_jaxopt(r_j, th_j, log_x0):
    """Single LBFGS run for one feature, one init. Returns (log_params, fun_val)."""
    def obj(log_params):
        return _neg_ll_one_feature(log_params, r_j, th_j)
    solver = jaxopt.LBFGS(fun=obj, maxiter=200, tol=1e-6)
    result = solver.run(log_x0)
    return result.params, result.state.value  # .value not .fun_val

fit_one_jit = jax.jit(fit_one_jaxopt)

# Test on feature j=0, one init
r0 = jnp.array(ratings[:, 0])
th0 = jnp.full(N, float(pz1_mean[0]))
log_x0 = _LOG_INITS[0]

params_jax, fval_jax = fit_one_jit(r0, th0, log_x0)
print(f"jaxopt params (log-space): {np.array(params_jax).round(3)}")
print(f"jaxopt fun val: {fval_jax:.4f}")

# Compare to scipy on same init
def f_and_g(lp):
    val, grad = _neg_ll_and_grad(jnp.array(lp), r0, th0)
    return float(val), np.array(grad)
res_sp = scipy_minimize(f_and_g, np.array(log_x0), jac=True, method='BFGS')
print(f"\nscipy  params (log-space): {res_sp.x.round(3)}")
print(f"scipy  fun val: {res_sp.fun:.4f}")
print(f"\nDifference in fun val: {abs(fval_jax - res_sp.fun):.6f}")

jaxopt params (log-space): [0.524 1.539 1.949 0.819]
jaxopt fun val: -5.7950

scipy  params (log-space): [1.307 0.581 1.155 2.738]
scipy  fun val: -9.3760

Difference in fun val: 3.581039


## 4. Multi-start with jaxopt: vmap over initializations
Replace the Python loop over `_LOG_INITS` with vmap.

In [5]:
def fit_one_jaxopt_multistart(r_j, th_j):
    """Best-of-inits fit for one feature using vmap over LOG_INITS."""
    # vmap over the 4 starting points
    params_all, fvals_all = jax.vmap(
        lambda log_x0: fit_one_jaxopt(r_j, th_j, log_x0)
    )(_LOG_INITS)  # (4, 4), (4,)
    best_idx = jnp.argmin(fvals_all)
    return params_all[best_idx]  # (4,) best log_params

fit_multistart_jit = jax.jit(fit_one_jaxopt_multistart)

t0 = time.time()
best_log_params = fit_multistart_jit(r0, th0)
best_log_params.block_until_ready()
t1 = time.time()
# Second call (compiled)
best_log_params = fit_multistart_jit(r0, th0)
best_log_params.block_until_ready()
t2 = time.time()

print(f"First call (includes compile): {t1-t0:.3f}s")
print(f"Second call (compiled):        {t2-t1:.4f}s")
print(f"Best log_params: {np.exp(np.array(best_log_params)).round(3)}  (in param space)")

First call (includes compile): 3.403s
Second call (compiled):        0.0623s
Best log_params: [ 3.696  1.788  3.174 15.454]  (in param space)


## 5. vmap over S MCMC samples — the key test
For one feature j, run S=200 separate fits (one per MCMC sample of pz1),
all in parallel via vmap.

In [6]:
def fit_one_sample(th_scalar, r_j):
    """Fit Beta mixture for one feature with one scalar pz1 value."""
    th_j = jnp.full(r_j.shape[0], th_scalar)  # broadcast scalar to (N,)
    return fit_one_jaxopt_multistart(r_j, th_j)  # (4,) log_params

# vmap over S samples for feature j=0
fit_over_samples = jax.vmap(fit_one_sample, in_axes=(0, None))
fit_over_samples_jit = jax.jit(fit_over_samples)

pz1_samples_j0 = jnp.array(pz1_samples[:, 0])  # (S,)
r0_jnp = jnp.array(ratings[:, 0])               # (N,)

t0 = time.time()
log_params_all_s = fit_over_samples_jit(pz1_samples_j0, r0_jnp)
log_params_all_s.block_until_ready()
t1 = time.time()
# Second call
log_params_all_s = fit_over_samples_jit(pz1_samples_j0, r0_jnp)
log_params_all_s.block_until_ready()
t2 = time.time()

print(f"vmap over S={S} samples, feature j=0:")
print(f"  First call  (compile+run): {t1-t0:.3f}s")
print(f"  Second call (compiled):    {t2-t1:.3f}s")
print(f"  Output shape: {log_params_all_s.shape}  (S, 4) log_params")

vmap over S=200 samples, feature j=0:
  First call  (compile+run): 11.407s
  Second call (compiled):    6.504s
  Output shape: (200, 4)  (S, 4) log_params


## 6. vmap over both S samples AND J features
Full Option 3: S × J fits in one vectorized call.

In [7]:
def fit_one_sample_all_features(pz1_s, ratings):
    """For one MCMC sample (J,), fit Beta for all J features. Returns (J, 4) log_params."""
    # vmap over J features
    return jax.vmap(fit_one_sample, in_axes=(0, 1))(pz1_s, ratings)  # (J, 4)

# vmap over S samples
fit_all = jax.vmap(fit_one_sample_all_features, in_axes=(0, None))
fit_all_jit = jax.jit(fit_all)

pz1_samp_jnp = jnp.array(pz1_samples)  # (S, J)
ratings_jnp  = jnp.array(ratings)       # (N, J)

t0 = time.time()
log_params_SJ = fit_all_jit(pz1_samp_jnp, ratings_jnp)
log_params_SJ.block_until_ready()
t1 = time.time()
log_params_SJ = fit_all_jit(pz1_samp_jnp, ratings_jnp)
log_params_SJ.block_until_ready()
t2 = time.time()

print(f"vmap over S={S} samples × J={J} features:")
print(f"  First call  (compile+run): {t1-t0:.3f}s")
print(f"  Second call (compiled):    {t2-t1:.3f}s")
print(f"  Output shape: {log_params_SJ.shape}  (S, J, 4)")

vmap over S=200 samples × J=5 features:
  First call  (compile+run): 32.984s
  Second call (compiled):    28.347s
  Output shape: (200, 5, 4)  (S, J, 4)


## 7. Option 3 log likelihood
For each of the S fits, evaluate log P(ratings | pz1_s, beta_params_s),
then logsumexp over S to get the marginal log likelihood.

In [8]:
def log_lik_one_sample(log_params_j, pz1_s_j, r_j):
    """Log lik for one feature, one sample. Scalar."""
    th_j = jnp.full(r_j.shape[0], pz1_s_j)
    return -_neg_ll_one_feature(log_params_j, r_j, th_j)

def log_lik_one_sample_all_features(log_params_J, pz1_s, ratings):
    """Sum log lik over J features for one MCMC sample. Scalar."""
    # vmap over J
    per_feature = jax.vmap(log_lik_one_sample, in_axes=(0, 0, 1))(
        log_params_J, pz1_s, ratings
    )  # (J,)
    return jnp.sum(per_feature)

# vmap over S
log_liks_all = jax.vmap(log_lik_one_sample_all_features, in_axes=(0, 0, None))(
    log_params_SJ, pz1_samp_jnp, ratings_jnp
)  # (S,)

# Option 3: logsumexp over S samples
ll_option3 = float(logsumexp(np.array(log_liks_all)) - np.log(S))

print(f"Log likelihoods per sample: min={log_liks_all.min():.1f}, max={log_liks_all.max():.1f}, mean={log_liks_all.mean():.1f}")
print(f"\nOption 1 log lik (scipy, mean pz1):     {ll_scipy:.2f}")
print(f"Option 3 log lik (jaxopt vmap, logsumexp): {ll_option3:.2f}")
print(f"\nDifference: {ll_option3 - ll_scipy:.2f}  (Option 3 - Option 1)")

Log likelihoods per sample: min=49.7, max=74.4, mean=66.2

Option 1 log lik (scipy, mean pz1):     52.91
Option 3 log lik (jaxopt vmap, logsumexp): 70.94

Difference: 18.03  (Option 3 - Option 1)


## 8. Benchmark: scale up to J=15 (real problem size)
Test how long the compiled vmap takes at the actual number of features.

In [9]:
J_real = 15
S_real = 200  # subsampled from 1999

rng2 = np.random.default_rng(1)
ratings_real   = np.clip(rng2.beta(3, 3, size=(N, J_real)), 1e-6, 1-1e-6)
pz1_samp_real  = rng2.beta(5, 3, size=(S_real, J_real))

ratings_real_jnp  = jnp.array(ratings_real)
pz1_samp_real_jnp = jnp.array(pz1_samp_real)

# Compile
t0 = time.time()
lp_real = fit_all_jit(pz1_samp_real_jnp, ratings_real_jnp)
lp_real.block_until_ready()
t1 = time.time()
# Compiled run
lp_real = fit_all_jit(pz1_samp_real_jnp, ratings_real_jnp)
lp_real.block_until_ready()
t2 = time.time()

print(f"J={J_real}, S={S_real}, N={N}")
print(f"  First call  (compile+run): {t1-t0:.2f}s")
print(f"  Second call (compiled):    {t2-t1:.3f}s")

# Compare: how long would scipy loop take for S=200 fits?
from model_jax import _fit_one_feature
t0 = time.time()
for s in range(min(5, S_real)):  # just 5 to estimate
    pz1_s = jnp.array(pz1_samp_real[s])  # (J,)
    pz1_s_bc = jnp.tile(pz1_s, (N, 1))
    _ = fit_beta_mixtures_all_features(ratings_real_jnp, pz1_s_bc)
t_per_sample = (time.time() - t0) / 5
print(f"\nScipy: ~{t_per_sample:.3f}s per sample → {t_per_sample*S_real:.1f}s for S={S_real}")
print(f"jaxopt vmap speedup: ~{t_per_sample*S_real / (t2-t1):.0f}×")

J=15, S=200, N=100
  First call  (compile+run): 85.30s
  Second call (compiled):    78.644s

Scipy: ~0.495s per sample → 99.1s for S=200
jaxopt vmap speedup: ~1×
